In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys
import logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Percorsi
sidecar_path = project_root / "data/processed/armstrong/sc_multi_pret_distanceclasses1.json"
queries_path = project_root / "data/queries/armstrong/eval_queries.json"

# Inizializzazione gestore pipeline
handler = PipelineHandler(
    sidecar_path=sidecar_path,
    queries_path=queries_path
)

print("!>> PipelineHandler e SidecarManager inizializzati con successo.")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
07:12:19 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
07:12:23 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> PipelineHandler e SidecarManager inizializzati con successo.


In [10]:
### 1 ### Impostazione Parametri
handler.set_parameters(
    top_j=5,
    top_k=7,
    top_n=5,
    order_by=["DIST", "TAGS"]
)

08:01:34 [INFO] src.pipeline_handler: ?> Parametri aggiornati: top_j=5, top_k=7, top_n=5, order_by=['DIST', 'TAGS']


In [11]:
### 2 ### Selezione Query

#Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
handler.update_query(new_id="Q_AMBIG_01")

#handler.update_query(new_text="Quali sono i contributi musicali di Louis Armstrong e che strumento suonava?")

08:01:34 [INFO] src.pipeline_handler: ?> Testo Query aggiornato: /home/jovyan/tesi_graphrag/data/queries/armstrong/eval_queries.json--ID>Q_AMBIG_01 
08:01:35 [INFO] src.pipeline_handler: ?> Vettore embedding Query aggiornato: text:Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?


In [12]:
### 3 ### Tag query
conv_tag_list = ["Astronautica"]
ass_tag_list = [""]
auto_ass_flag = True

handler.update_query_tags(
    new_conv_list=conv_tag_list,
    new_ass_list=ass_tag_list,
    auto_ass_flag=auto_ass_flag,
    print_debug=True
)

08:01:38 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Jazz
DB>> expanded_tag:
  >>>Jazz: "Jazz è una musica d'origine afroamericana caratterizzata da ritmi e melodie complesse, che spesso combina elementi di blues, swing e bebop, con strumenti come il saxofono, la tromba, il piano e la batteria, utilizzati in combo con cantanti come Billie Holiday, Ella Fitzgerald e Louis Armstrong."


08:01:50 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Ciclismo
DB>> expanded_tag:
  >>>Ciclismo: "Ciclismo: è una disciplina sportiva che combina la tecnica di guida con l'equilibrio e la resistenza, caratterizzata da biciclette specializzate in diverse categorie come il ciclismo su strada, il ciclismo su pista, il mountain bike, il BMX, lo skatepark, il ciclismo di montagna, il ciclismo di strada, il ciclismo su pista, il ciclismo su pista a squadre e il ciclismo di staza. Le componenti chiave includono le ruote, i freni, la sospensione, l'ergometria, gli pneumatici, le catene, i pedali, le maniglie, le gomme, le ruote da corsa, le ruote da mountain bike, le ruote da BMX e le ruote da skatepark. Le entità correlate includono i ciclisti, le squadre di ciclismo, gli eventi di ciclismo, le competizioni di ciclismo, le gare di ciclismo, le tappe di ciclismo, le classifiche di ciclismo e le statistiche di ciclismo."


08:02:21 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Astronautica
DB>> expanded_tag:
  >>>Astronautica: 'Astronautica: l'astronautica è lo studio, la progettazione, la costruzione e l'esplorazione degli spazi aereo e spaziale, che comprende sotto-categorie come la cosmonautica, la spaceflight, la missione spaziale, la navigazione spaziale, la propulsione spaziale, la struttura spaziale, la protezione spaziale, l'astronomia, la geofisica, la meteorologia, la radiotecnica, la comunicazione spaziale, la telemetria, la navigazione GPS, la propulsione a razzo, la propulsione ionica, la propulsione a idrogeno, la struttura in titanio, la protezione con blindature, la protezione con materiali compositi, l'astronauta, il cosmonauta, la spacewalk, la manovra spaziale, la navigazione spaziale, la comunicazione radio, la telemetria, la navigazione GPS, la propulsione a razzo, la propulsione ionica, la propulsione a idrogeno, la struttura in titanio, la protezione con blindature, la protezione con materiali compositi.'


08:02:50 [INFO] src.pipeline_handler: ?> Query vettorizzata. Tag assegnati: ['Astronautica']


!>> Tag assegnati alla query:
   >>> Tag di Conversazione: ['Astronautica']
   >>> Tag Manuali della Query: []
   >>> Tag Automaticamente assegnati alla Query: []
       |> Tag Globali disponibili: ['Jazz', 'Ciclismo', 'Astronautica']
<<< Lista completa: ['Astronautica']


In [13]:
### 4 ### Retrieval Vettoriale
handler.run_vectorial_retrieval()

08:02:50 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points/query "HTTP/1.1 200 OK"
08:02:50 [INFO] src.pipeline_handler: ?> Recuperati 7 record con relativi vettori di embedding


In [14]:
### 5 ### Reranking
handler.run_top_j_distance_tag_reranking(print_debug=True)

08:02:51 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points "HTTP/1.1 200 OK"


?> [DEBUG INFO] all_reranked:
         >> Il Reranker ha restituito 10 record
    >>> cid: ['neil_armstrong_1']
        |> category: [1]
           |> final_score: [0.615]
    >>> cid: ['neil_armstrong_7']
        |> category: [1]
           |> final_score: [0.5872]
    >>> cid: ['neil_armstrong_2']
        |> category: [2]
           |> final_score: [0.8131]
    >>> cid: ['neil_armstrong_3']
        |> category: [2]
           |> final_score: [0.7134]
    >>> cid: ['louis_armstrong_4']
        |> category: [3]
           |> final_score: [0.5447]
    >>> cid: ['lance_armstrong_2']
        |> category: [4]
           |> final_score: [0.6698]
    >>> cid: ['louis_armstrong_6']
        |> category: [4]
           |> final_score: [0.6627]
    >>> cid: ['lance_armstrong_1']
        |> category: [4]
           |> final_score: [0.6303]
    >>> cid: ['lance_armstrong_3']
        |> category: [4]
           |> final_score: [0.5996]
    >>> cid: ['louis_armstrong_1']
        |> category: [4]
   

In [15]:
import ipywidgets as widgets
from IPython.display import display

print("I Grafi che contano veramente nella pipeline finale (dunque escludendo quelli presenti per DEBUG) sono:")
print("  > 1° Retrieval (Vettoriale)")
print("  > Top-J Reranked")

# Mappatura tra i nomi visibili nei pulsanti e le chiavi interne delle fasi
PHASE_MAP = {
    "1° Retrieval (Vettoriale)": "vectorial",
    "2° Retrieval (Distanze)": "distance",
    "Top-J Reranked": "reranked",
    "All Reranked": "all_reranked"
}

# Selettore orizzontale delle fasi (ToggleButtons)
phase_selector = widgets.ToggleButtons(
    options=list(PHASE_MAP.keys()),
    description='Fase Grafo:',
    disabled=False,
    button_style='info',
    tooltips=[
        'Mostra i Top-K nodi del retrieval vettoriale iniziale',
        'Mostra i nodi iniziali uniti ai vicini recuperati per distanza',
        'Mostra i Top-J nodi selezionati dal Reranker per l\'LLM',
        'Mostra l\'intero elenco ordinato dal Reranker'
    ],
)

# Container per il rendering dinamico del widget
output_area = widgets.Output()

def on_phase_change(change):
    selected_label = change['new']
    phase_key = PHASE_MAP[selected_label]
    
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(phase_key)
        display(handler.widget)

# Registrazione dell'evento sul cambio di selezione
phase_selector.observe(on_phase_change, names='value')

# Inizializzazione della vista sulla prima fase
handler.update_widget_view("vectorial")

# Visualizzazione dell'interfaccia a schermo
display(phase_selector)
display(output_area)

# Popola l'output area con la vista iniziale del grafo
with output_area:
    display(handler.widget)

08:02:51 [INFO] src.pipeline_handler: ?> Widget aggiornato alla fase 'vectorial' (7 nodi)


I Grafi che contano veramente nella pipeline finale (dunque escludendo quelli presenti per DEBUG) sono:
  > 1° Retrieval (Vettoriale)
  > Top-J Reranked


ToggleButtons(button_style='info', description='Fase Grafo:', options=('1° Retrieval (Vettoriale)', '2° Retrie…

Output()

In [16]:
### LLM ### Generazione risposte LLM
comparison = handler.compare_retrieval_vs_reranking()

baseline = comparison["vectorial_baseline"]
reranked = comparison["post_reranking"]

print("=" * 80)
print(f"???> Query: '{baseline['query']}'")
print("=" * 80)

print("\n --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---")
print(f"  >> Chunk inclusi ({len(baseline['chunk_ids'])}): {baseline['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(baseline["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(baseline["response"])

print("\n" + "=" * 80)

print("\n --- 2. RISPOSTA RERANKED (Post DistanceTagReranking / Top-J) ---")
print(f"  >> Chunk inclusi ({len(reranked['chunk_ids'])}): {reranked['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(reranked["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(reranked["response"])
print("=" * 80)

08:02:51 [INFO] src.pipeline_handler: ?> Invocazione LLM
08:02:51 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
08:03:17 [INFO] src.pipeline_handler: ?> Invocazione LLM
08:03:18 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


???> Query: 'Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?'

 --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---
  >> Chunk inclusi (7): ['d193d93c-843b-5d0f-bffb-e26d2eed183e', 'e6817b97-dfcf-5b4c-b301-072360c9723b', 'c24a84d9-28b1-56c9-afdf-129af82eb17b', '8768da0f-ca2a-5162-8afa-71ebfc3dfcdc', 'ac381e01-85a8-5a9b-a32c-365353d16b46', '615bbd74-c451-591e-b8d2-82890df6f35d', '106100c7-1928-5364-873c-16830ebc9b7f']

  >> PROMPT INVIATO:
----------------------------------------
Sei un assistente di ricerca. Rispondi alla domanda seguente basandoti ESCLUSIVAMENTE sul contesto fornito. Se il contesto non contiene informazioni sufficienti o è parziale, evidenzialo chiaramente senza inventare fatti non presenti nelle fonti.

Domanda: Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?

Contesto recuperato (7 chunk):
[CHUNK 1 | ID: d193d93c-843b-5d0f-bffb-e26d2eed183e | Score: 0.6698]
Nel mondo del ciclismo pr